# AutoIntake — Parameter Sweep

Questo notebook calcola la trasmissione (`W_0`) per un grande numero di combinazioni dei parametri `(a, b, c)` e li salva in un CSV.

**Funzionalità chiave:**
- Ad ogni avvio controlla il CSV esistente e riprende dal punto in cui si era fermato.
- Può essere interrotto e riavviato in qualsiasi momento senza perdere dati.
- I risultati vengono scritti sul CSV immediatamente dopo ogni calcolo.
- Compatibile con Google Colab (mount su Google Drive per persistenza).

## 1. Setup Google Drive (solo su Colab)

Monta Google Drive per salvare i risultati in modo persistente tra una sessione e l'altra.
Se non sei su Colab, salta questa cella.

In [ ]:
import sys

ON_COLAB = 'google.colab' in sys.modules

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Cartella base su Drive dove salvare tutto
    DRIVE_BASE = '/content/drive/MyDrive/AutoIntake'
    import os
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f'Google Drive montato. Cartella di lavoro: {DRIVE_BASE}')
else:
    DRIVE_BASE = '.'
    print('Esecuzione locale. Cartella di lavoro corrente.')

## 2. Installazione dipendenze

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip_install('numpy-stl')
pip_install('trimesh')
pip_install('scipy')
pip_install('matplotlib')
pip_install('pandas')
pip_install('mpi4py')

print('Dipendenze installate.')

## 3. Clona/carica il repository

Su Colab clona il repository da GitHub (sostituisci l'URL con il tuo).
Se sei in locale questa cella è da saltare.

In [ ]:
import os, sys

ON_COLAB = 'google.colab' in sys.modules

if ON_COLAB:
    REPO_DIR = '/content/AutoIntake'

    if not os.path.isdir(REPO_DIR):
        GITHUB_URL = 'https://github.com/mattiademartino/AutoIntake.git'
        subprocess.check_call(['git', 'clone', GITHUB_URL, REPO_DIR])
        print(f'Repository clonato in {REPO_DIR}')
    else:
        subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])
        print(f'Repository aggiornato in {REPO_DIR}')

    # Compila l'estensione C rarfast se necessario
    rarfast_so = os.path.join(REPO_DIR, 'SMARTA_functions', 'rarfast.cpython-310-x86_64-linux-gnu.so')
    if not os.path.isfile(rarfast_so):
        print('Compilazione rarfast.c ...')
        subprocess.check_call([
            sys.executable, 'setup.py', 'build_ext', '--inplace'
        ], cwd=os.path.join(REPO_DIR, 'SMARTA_functions'))
        print('Compilazione completata.')

else:
    REPO_DIR = os.path.abspath('.')   # directory corrente in locale

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Working directory: {os.getcwd()}')

## 4. Configurazione: griglia dei parametri e percorso CSV

Definisci qui i range di `a`, `b`, `c` da esplorare.
Il numero totale di valutazioni sarà `N_a × N_b × N_c`.

In [ ]:
import numpy as np
import os, sys

# ---------------------------------------------------------------------------
# Griglia dei parametri
# ---------------------------------------------------------------------------
A_VALUES = np.linspace(-1.0, 1.0, 30)
B_VALUES = np.linspace(-1.0, 1.0, 30)
C_VALUES = np.linspace(-1.0, 1.0, 30)
# ---------------------------------------------------------------------------

# Percorso del CSV dei risultati — su Colab viene salvato su Drive
ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    CSV_PATH = os.path.join(DRIVE_BASE, 'result.csv')
else:
    CSV_PATH = os.path.join(os.getcwd(), 'result.csv')

# Costruisci l'elenco completo di combinazioni (a, b, c)
all_combinations = [
    (float(a), float(b), float(c))
    for a in A_VALUES
    for b in B_VALUES
    for c in C_VALUES
]

total = len(all_combinations)
print(f'Griglia: {len(A_VALUES)} x {len(B_VALUES)} x {len(C_VALUES)} = {total:,} combinazioni totali')
print(f'CSV dei risultati: {CSV_PATH}')

## 5. Funzioni di utilità: CSV e ripresa del calcolo

In [ ]:
import csv


def load_completed(csv_path):
    """Carica le combinazioni (a, b, c) già calcolate dal CSV.
    Restituisce un set di tuple (a_str, b_str, c_str) già presenti.
    """
    completed = set()
    if not os.path.isfile(csv_path):
        return completed
    with open(csv_path, 'r', newline='') as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) >= 4:
                # Usa la rappresentazione stringa raw per confronto esatto
                completed.add((row[0], row[1], row[2]))
    return completed


def is_completed(a, b, c, completed_set):
    """Controlla se (a, b, c) è già presente nel set dei completati."""
    return (str(a), str(b), str(c)) in completed_set


def append_result(csv_path, a, b, c, w):
    """Aggiunge una riga al CSV immediatamente (flush per sicurezza)."""
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([a, b, c, w])
        f.flush()
        os.fsync(f.fileno())


def get_remaining(all_combinations, completed_set):
    """Ritorna le combinazioni non ancora calcolate."""
    remaining = [
        (a, b, c) for (a, b, c) in all_combinations
        if not is_completed(a, b, c, completed_set)
    ]
    return remaining


# Mostra quante combinazioni sono già state calcolate
completed_set = load_completed(CSV_PATH)
remaining = get_remaining(all_combinations, completed_set)

print(f'Già calcolati:  {len(completed_set)}')
print(f'Da calcolare:   {len(remaining)}')
print(f'Totale:         {total}')

## 6. Inizializzazione geometria fissa (inlet, outlet, L)

Queste operazioni vengono fatte una sola volta per sessione.

In [ ]:
import numpy as np
from stl import mesh as stl_mesh_lib
from create_faces import get_inlet, get_outlet, generate_structured_points

L_INTAKE = 80.0

def calculate_L():
    """Calcola la lunghezza efficace dell'intake sottraendo l'altezza del honeycomb."""
    m = stl_mesh_lib.Mesh.from_file('Honeycombs/HC1.STL')
    points_z = m.points.reshape(-1, 3)[:, 2]
    return float(np.max(points_z) - np.min(points_z))

# Crea cartella mesh se non esiste
os.makedirs('mesh', exist_ok=True)

HC_HEIGHT = calculate_L()
L = L_INTAKE - HC_HEIGHT

print(f'Altezza honeycomb: {HC_HEIGHT:.4f} mm')
print(f'Lunghezza intake efficace L = {L:.4f} mm')

# Genera inlet e outlet (fissi, non dipendono da a,b,c)
get_inlet(verbose=False)
get_outlet(L, verbose=False)

print('Inlet e outlet generati.')

## 7. Loop principale — calcolo e salvataggio

Per ogni combinazione rimanente:
1. Genera la geometria delle facce (`face0.stl` … `face3.stl`).
2. Esegue la simulazione SMARTA.
3. Salva il risultato sul CSV immediatamente.

Puoi interrompere (`Runtime → Interrupt execution`) e rilanciare: il notebook riprenderà dall'ultima riga mancante nel CSV.

In [ ]:
from SMARTA import get_trasmission
from create_faces import generate_structured_points, rotate_points, create_stl_from_points
import traceback


def compute_transmission(a, b, c, L):
    """Genera la geometria e lancia la simulazione SMARTA per (a, b, c, L)."""
    # Genera i punti della superficie parametrica
    points = generate_structured_points(a, b, c, L, verbose=False)

    # Crea le 4 facce ruotate (0°, 90°, 180°, 270°)
    for angle in [0, 90, 180, 270]:
        rotated = rotate_points(points, angle)
        create_stl_from_points(rotated, filename=f'mesh/face{angle // 90}.stl', verbose=False)

    # Esegui la simulazione
    return get_trasmission(verbose=False)


# Ricarica lo stato aggiornato del CSV prima di partire
completed_set = load_completed(CSV_PATH)
remaining = get_remaining(all_combinations, completed_set)

n_remaining = len(remaining)
n_done = 0

print(f'Inizio sweep: {n_remaining} combinazioni da calcolare su {total} totali.\n')

for idx, (a, b, c) in enumerate(remaining):
    print(f'[{idx + 1}/{n_remaining}]  a={a:.4f}  b={b:.4f}  c={c:.4f}  ...', end='  ', flush=True)

    try:
        w = compute_transmission(a, b, c, L)
        append_result(CSV_PATH, a, b, c, w)
        n_done += 1
        print(f'W_0 = {w:.6e}')
    except Exception as e:
        print(f'ERRORE: {e}')
        traceback.print_exc()
        # Non salva la riga — verrà ritentata al prossimo avvio

print(f'\nSweep completato. Calcolati {n_done} nuovi valori.')
print(f'CSV aggiornato: {CSV_PATH}')

## 8. Visualizzazione dei risultati

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

if not os.path.isfile(CSV_PATH):
    print('CSV non trovato, esegui prima il loop di calcolo.')
else:
    df = pd.read_csv(CSV_PATH, header=None, names=['a', 'b', 'c', 'W_0'])
    df = df.dropna()
    print(f'Righe nel CSV: {len(df)}')
    print(df.describe())

    # --- Scatter plot W_0 vs indice ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, param in zip(axes, ['a', 'b', 'c']):
        ax.scatter(df[param], df['W_0'], s=10, alpha=0.6)
        ax.set_xlabel(param)
        ax.set_ylabel('W_0')
        ax.set_title(f'W_0 vs {param}')
        ax.grid(True)
    plt.tight_layout()
    plt.show()

    # --- Heatmap per coppia (a, b) fissando c al valore più comune ---
    if len(df) > 0:
        c_fixed = df['c'].mode()[0]
        sub = df[np.isclose(df['c'], c_fixed)].copy()
        if len(sub) > 1:
            pivot = sub.pivot_table(index='a', columns='b', values='W_0', aggfunc='mean')
            fig2, ax2 = plt.subplots(figsize=(7, 5))
            im = ax2.imshow(
                pivot.values,
                aspect='auto',
                origin='lower',
                extent=[pivot.columns.min(), pivot.columns.max(),
                        pivot.index.min(), pivot.index.max()],
                cmap='viridis'
            )
            plt.colorbar(im, ax=ax2, label='W_0')
            ax2.set_xlabel('b')
            ax2.set_ylabel('a')
            ax2.set_title(f'Heatmap W_0(a, b)  —  c = {c_fixed:.3f}')
            plt.tight_layout()
            plt.show()